In [3]:
pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def categorizador():
    """Ejecuta la inferencia de la red neuronal para categorizar productos."""
    try:
        import tensorflow as tf
        from tensorflow.keras.utils import pad_sequences
    except ImportError:
        logger.error("Tensorflow no instalado.")
        return

    logger.info("Iniciando proceso de categorización neuronal...")
    engine = get_db_engine()
    if not engine: return

    try:
        # Obtener productos sin categoría
        query = """
            SELECT tp.csku, tp.cnombre, tp.cdescripcion, tp.cmarca, tnc.nid as id_subcategoria
            FROM tbl_producto AS tp
            LEFT JOIN tbl_subcategoria AS tnc ON tnc.nid = tp.nid_subcategoria
            WHERE tp.nid_subcategoria IS NULL
        """
        df_prod = pd.read_sql(query, engine)
        
        if df_prod.empty:
            logger.info("No hay productos pendientes de categorización.")
            return

        # Rutas dinámicas
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
        RN_DIR = os.path.join(BASE_DIR, "Red_neuronal")
        
        path_model = os.path.join(RN_DIR, "modelo_categorias_optimizado.keras")
        path_tok = os.path.join(RN_DIR, "tokenizer.pkl")
        path_enc = os.path.join(RN_DIR, "labelencoder.pkl")

        if not os.path.exists(path_model):
            raise FileNotFoundError(f"Modelo no encontrado: {path_model}")

        # Cargar artefactos
        model = tf.keras.models.load_model(path_model)
        with open(path_tok, "rb") as f: tokenizer = pickle.load(f)
        with open(path_enc, "rb") as f: le = pickle.load(f)

        # Preprocesamiento
        df_prod["texto"] = (
            df_prod["cnombre"].fillna("") + " " + 
            df_prod["cdescripcion"].fillna("") + " " + 
            df_prod["cmarca"].fillna("")
        )
        df_prod["texto"] = df_prod["texto"].apply(normalize_text)
        
        seq = tokenizer.texts_to_sequences(df_prod["texto"])
        X = pad_sequences(seq, maxlen=200)

        # Inferencia
        preds = model.predict(X, verbose=0)
        y_classes = np.argmax(preds, axis=1)
        categorias = le.inverse_transform(y_classes)
        df_prod["categoria_predicha"] = categorias

        # Obtener IDs de subcategorías
        df_sub = pd.read_sql("SELECT nid as nid_subcategoria, cnombre_subcategoria as nombre_subcategoria FROM tbl_subcategoria", engine)
        
        df_final = df_prod.merge(df_sub, left_on="categoria_predicha", right_on="nombre_subcategoria", how="left")
        
        # Update en Batch
        updates = list(zip(df_final["id_subcategoria"], df_final["csku"]))
        updates = [(int(cat), sku) for cat, sku in updates if pd.notna(cat)]

        if updates:
            with engine.connect() as conn:
                with conn.connection.cursor() as cursor:
                    execute_batch(cursor, "UPDATE tbl_producto SET nid_subcategoria  = %s WHERE csku = %s", updates)
                    conn.connection.commit()
            logger.info(f"Categorizados {len(updates)} productos.")

    except Exception as e:
        logger.error(f"Error en categorizador: {e}")
    finally:
        engine.dispose()
        tf.keras.backend.clear_session()
        gc.collect()
